# Radon vector-deskew lab

Reads a **manual label file**, pulls each label's **exact vectors** (via
`LabelEntry.vector_signatures` -> `label_schema.path_signature`), and sweeps a Radon-style
angle `theta` **purely on vector geometry** -- no rasterization anywhere.

### The projection (one fixed function, `project(vectors, theta)`)

1. rotate every vector's points by `-theta` about the cluster centre
2. take the cluster's rotated-bbox `[X0, X1] x [Y0, Y1]`
3. division lines **scale with the rotated-bbox height** `H = Y1 - Y0`: fixed spacing
   `LINE_SPACING` pt -> `n = clip(round(H / LINE_SPACING), 2, MAX_LINES)` internal
   **horizontal** lines `y_k = Y0 + k*H/(n+1)`, `k = 1..n` (first / last edge excluded).
   So a taller rotated bbox gets more lines.
4. for each line `y = y_k`, count how many vectors it **geometrically intersects**
   (`l` -> segment, `c` -> Bernstein-flattened sub-segments, `re`/`qu` -> the 4 rotated
   edges); each vector counts 0 or 1
5. -> a length-`n` array = the 1-D projection / histogram at that `theta`

The rotated-bbox **width `W` and height `H`** are also recorded at every `theta`
(cell 9) -- they themselves carry a deskew signal.

### Value functions (`VALUES`, pluggable)

`theta` only sweeps a 90-degree range (`THETA_MIN..THETA_MIN+90`); every `theta` is paired with
its perpendicular companion `theta + 90`, so each value function sees **both** projections at
once: `fn(a, b) -> float`, where `a = project(vectors, theta)` and `b = project(vectors,
theta + 90)`. E.g. `theta=-90` pairs with `0`, `theta=-75` pairs with `15`, etc. Add one by
appending to `VALUES` (and `OPT` with `"max"` / `"min"` for the argopt marker), then re-run from
the sweep cell.


## 0 - Config

In [ ]:
from pathlib import Path

LABEL_PATH        = None     # None -> first (name-sorted) outputs/labels/*.json
PDF_PATH_OVERRIDE = None     # None -> LabelSet.pdf_path (resolved vs repo root)
PAGE_INDEX        = None     # None -> every page present in the label file

# `theta` sweeps a 90-degree range only -- each theta is paired with its perpendicular
# companion `theta + 90` (see cell 12), so THETA_MAX should stay THETA_MIN + 90.
THETA_MIN, THETA_MAX, THETA_STEP = -90.0, 0.0, 2.0
LINE_SPACING   = 1.0         # pt between division lines -> line count scales with bbox height
MAX_LINES      = 600         # safety cap on lines per projection
CURVE_SAMPLES  = 24          # sub-segments a cubic 'c' item is flattened into
GRID_THETA_COLS = 9          # theta columns in the per-cluster grid


def n_lines_for(height):
    """Division-line count for a rotated-bbox height (scales with the bbox)."""
    return int(np.clip(round(height / LINE_SPACING), 2, MAX_LINES))

## 1 - Path bootstrap

In [ ]:
import os, sys
_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.append(_root)

## 2 - Labels -> the exact labelled vector set per label

`extract_vectors(page)` once per page (raw `get_drawings()` geometry -- no classification,
no clustering, no rendering), keyed by `path_signature`; each label pulls its own vectors by
the signatures it stored.

In [ ]:
import numpy as np
from rastervec.Reader.reader import Reader
from rastervec.pipelines._steps import extract_vectors
from rastervec.Evaluation.Labelling.label_schema import (
    load_labels, split_labelset_by_source, path_signature)
from rastervec.helpers.geometry import union_bbox
from rastervec.paths import output_dir, REPO_ROOT


def _resolve_pdf(raw):
    p = Path(str(raw).replace(chr(92), "/"))
    if p.is_file():
        return p
    for base in (Path.cwd(), REPO_ROOT):
        for cand in ((base / p), (base / p.name)):
            if cand.is_file():
                return cand.resolve()
    for sub in ("references", "references2"):
        cand = REPO_ROOT / sub / p.name
        if cand.is_file():
            return cand
    raise FileNotFoundError(f"cannot locate PDF {raw!r} (cwd={Path.cwd()}, repo={REPO_ROOT})")


def _pick_label_path():
    if LABEL_PATH:
        return Path(LABEL_PATH)
    cands = sorted(Path(output_dir("labels")).glob("*.json"))
    if not cands:
        raise FileNotFoundError("no outputs/labels/*.json; set LABEL_PATH")
    return cands[0]


label_path = _pick_label_path()
labels = load_labels(label_path)
pdf_path = _resolve_pdf(PDF_PATH_OVERRIDE or labels.pdf_path)
manual = split_labelset_by_source(labels)["manual"].entries
print(f"label file: {label_path}")
print(f"pdf:        {pdf_path}")
print(f"manual entries: {len(manual)}")

by_page = {}
for e in manual:
    if PAGE_INDEX is None or e.page_index == PAGE_INDEX:
        by_page.setdefault(e.page_index, []).append(e)

matched = []          # (entry, [Vector, ...])
with Reader(pdf_path) as r:
    for pidx in sorted(by_page):
        page_vectors = extract_vectors(r.get_page(pidx))
        sigmap = {path_signature(v): v for v in page_vectors}
        for e in by_page[pidx]:
            sel = [sigmap[s] for s in e.vector_signatures if s in sigmap]
            matched.append((e, sel))
            print(f"  p{pidx}  {e.text!r:12}  rot={e.expected_rotation:>4}  "
                  f"found {len(sel)}/{len(e.vector_signatures)} vectors")

print(f"\nmatched clusters: {len(matched)}")

## 3 - Geometry helpers

In [ ]:
from rastervec.helpers.geometry import item_points


def rotate_pts(pts, theta_deg, centre):
    """Rotate (N,2) points by -theta about centre (radon-ray frame)."""
    t = np.deg2rad(-theta_deg)
    c, s = np.cos(t), np.sin(t)
    m = np.array([[c, -s], [s, c]])
    return (np.asarray(pts, float) - centre) @ m.T + centre


def _bezier(pts, n):
    p0, p1, p2, p3 = [np.asarray(p, float) for p in pts]
    u = np.linspace(0.0, 1.0, n + 1)[:, None]
    b = ((1 - u) ** 3 * p0 + 3 * (1 - u) ** 2 * u * p1
         + 3 * (1 - u) * u ** 2 * p2 + u ** 3 * p3)
    return list(zip(b[:-1], b[1:]))


def item_subsegments(item, curve_samples=None):
    """Straight (a, b) pieces approximating one Vector.items entry:
    'l' -> 1 segment, 'c' -> `curve_samples` Bernstein chords,
    're'/'qu' -> the 4 closed border edges."""
    n = CURVE_SAMPLES if curve_samples is None else curve_samples
    k = item[0]
    if k == "l":
        return [(item[1], item[2])]
    if k == "c":
        return _bezier(item_points(item), n)
    if k == "qu":
        c = [tuple(p) for p in item_points(item)]
    elif k == "re":
        x0, y0, x1, y1 = tuple(item[1])
        c = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]
    else:
        return []
    return list(zip(c, c[1:] + c[:1]))


def cluster_centre(vectors):
    x0, y0, x1, y1 = union_bbox([v.bbox for v in vectors])
    return np.array([(x0 + x1) / 2.0, (y0 + y1) / 2.0])


def rotated_segments(vectors, theta_deg):
    """Rotate the whole cluster by -theta about its centre.
    Returns (per_vector, (X0, X1, Y0, Y1)) where per_vector[i] is an
    (M_i, 2, 2) array of that vector's sub-segment endpoints in the rotated
    frame, and (X0, X1, Y0, Y1) is the cluster's rotated bbox."""
    centre = cluster_centre(vectors)
    per_vector = []
    for v in vectors:
        segs = [ab for it in v.items for ab in item_subsegments(it)]
        if not segs:
            per_vector.append(np.empty((0, 2, 2)))
            continue
        pts = rotate_pts(np.array(segs, float).reshape(-1, 2), theta_deg, centre)
        per_vector.append(pts.reshape(-1, 2, 2))
    have = [pv for pv in per_vector if pv.size]
    if not have:
        return per_vector, (0.0, 0.0, 0.0, 0.0)
    allpts = np.concatenate([pv.reshape(-1, 2) for pv in have])
    return per_vector, (float(allpts[:, 0].min()), float(allpts[:, 0].max()),
                        float(allpts[:, 1].min()), float(allpts[:, 1].max()))


def rotated_bbox_dims(vectors, theta_deg):
    """(width, height) of the cluster's rotated bbox at theta."""
    _, (X0, X1, Y0, Y1) = rotated_segments(vectors, theta_deg)
    return X1 - X0, Y1 - Y0

## 4 - The projection function

In [ ]:
def project(vectors, theta_deg):
    """(line_y[n], profile[n]) -- n horizontal rays y=y_k across the rotated
    cluster, n scaling with the rotated-bbox height; profile[k] = how many
    vectors ray k intersects."""
    per_vector, (X0, X1, Y0, Y1) = rotated_segments(vectors, theta_deg)
    H = Y1 - Y0
    n = n_lines_for(H)
    k = np.arange(1, n + 1)
    line_y = Y0 + k * H / (n + 1)
    prof = np.zeros(n)
    if H < 1e-9:
        return line_y, prof
    for pv in per_vector:
        if not pv.size:
            continue
        ylo = pv[:, :, 1].min(axis=1)[:, None]
        yhi = pv[:, :, 1].max(axis=1)[:, None]
        hit = ((ylo <= line_y[None, :]) & (line_y[None, :] <= yhi)).any(axis=0)
        prof += hit
    return line_y, prof

## 5 - Value functions  `fn(a[100], b[100]) -> float`

`a` is the profile at `theta`, `b` the profile at its perpendicular companion `theta + 90`.

In [ ]:
from rastervec.OCR import radon as R


def val_postl_sq(a, b):
    # TODO: combine with b (the theta+90 companion profile)
    return float(np.sum(np.asarray(a, float) ** 2))


def val_gap_objective(a, b):
    # TODO: combine with b (the theta+90 companion profile)
    return R._skew_objective(np.asarray(a, float))     # pure profile math, no raster


def val_variance(a, b):
    # TODO: combine with b (the theta+90 companion profile)
    return float(np.var(np.asarray(a, float)))


VALUES = {
    "postl_sq":      val_postl_sq,
    "gap_objective": val_gap_objective,
    "variance":      val_variance,
}
OPT = {"postl_sq": "max", "gap_objective": "min", "variance": "max"}

## 6 - Sweep **every** cluster in the label file

`sweep[ci]` holds that cluster's `entry`, `vectors`, per-theta `profiles`, and per-value
`curves` over the shared `thetas` grid.

In [ ]:
thetas = np.arange(THETA_MIN, THETA_MAX + THETA_STEP / 2, THETA_STEP)

sweep = []
for ci, (e, vec) in enumerate(matched):
    lines, profiles, pair_profiles, dims = {}, {}, {}, []
    for th in thetas:
        th = float(th)
        if vec:
            ly, pr_a = project(vec, th)
            _, pr_b = project(vec, th + 90.0)
            lines[th], profiles[th] = ly, pr_a
            pair_profiles[th] = (pr_a, pr_b)
            dims.append(rotated_bbox_dims(vec, th))
        else:
            dims.append((np.nan, np.nan))
    dims = np.array(dims, float)          # (n_theta, 2) = width, height
    curves = {n: np.array([fn(*pair_profiles[float(th)]) for th in thetas]) for n, fn in VALUES.items()} \
        if vec else {}
    sweep.append(dict(entry=e, vectors=vec, lines=lines, profiles=profiles,
                      pair_profiles=pair_profiles, dims=dims, curves=curves))
    print(f"#{ci:<3} {e.text!r:14} rot={e.expected_rotation:>4}  {len(vec)} vectors"
          + ("" if vec else "   <-- no vectors, skipped")
          + ("" if not vec else f"   lines/theta {min(len(v) for v in profiles.values())}"
             f"-{max(len(v) for v in profiles.values())}"))

## 7 - Per cluster: rotated-cluster render + 1-D projection at every theta

Two rows per cluster -- top: the cluster's vectors rotated by `-theta` (pure line render,
the 100 division lines dotted); bottom: the projection histogram at that theta.

In [ ]:
import matplotlib.pyplot as plt

sel = thetas[np.linspace(0, len(thetas) - 1, min(GRID_THETA_COLS, len(thetas))).astype(int)]
ncol = len(sel)

for ci, s in enumerate(sweep):
    vec, e = s["vectors"], s["entry"]
    if not vec:
        continue
    fig, axes = plt.subplots(2, ncol, figsize=(1.8 * ncol, 4.2), squeeze=False,
                             gridspec_kw=dict(height_ratios=[2, 1]))
    fig.suptitle(f"#{ci}  {e.text!r}   expected_rotation={e.expected_rotation}"
                 f"   ({len(vec)} vectors)", fontsize=10)
    for j, th in enumerate(sel):
        ti = int(np.argmin(np.abs(thetas - th)))
        prof = s["profiles"][float(th)]
        line_y = s["lines"][float(th)]
        per_vector, _ = rotated_segments(vec, float(th))
        w, h = s["dims"][ti]

        top = axes[0][j]
        for pv in per_vector:
            for (a, b) in pv:
                top.plot([a[0], b[0]], [a[1], b[1]], color="k", lw=0.5)
        for ly in line_y:
            top.axhline(ly, color="tab:blue", lw=0.3, alpha=0.35)
        top.set_aspect("equal"); top.invert_yaxis()
        top.set_xticks([]); top.set_yticks([])
        top.set_title(f"theta={th:+.0f}  {w:.0f}x{h:.0f}  ({len(prof)} ln)", fontsize=7)

        bot = axes[1][j]
        bot.bar(np.arange(len(prof)), prof, width=1.0)
        bot.set_xticks([]); bot.set_yticks([])
        bot.set_xlabel("\n".join(f"{n}={s['curves'][n][ti]:.3g}" for n in VALUES), fontsize=6)
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()

### 7b - (optional) interactive slider  — needs `ipywidgets`

In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display

    def _show(ci, ti):
        s = sweep[ci]
        if not s["vectors"]:
            print("no vectors for this cluster"); return
        th = float(thetas[ti])
        per_vector, _ = rotated_segments(s["vectors"], th)
        line_y = s["lines"][th]
        fig, (top, bot) = plt.subplots(2, 1, figsize=(11, 6),
                                       gridspec_kw=dict(height_ratios=[2, 1]))
        for pv in per_vector:
            for (a, b) in pv:
                top.plot([a[0], b[0]], [a[1], b[1]], color="k", lw=0.6)
        for ly in line_y:
            top.axhline(ly, color="tab:blue", lw=0.3, alpha=0.35)
        top.set_aspect("equal"); top.invert_yaxis()
        top.set_title(f"#{ci} {s['entry'].text!r}   theta={th:+.1f}   "
                      + "   ".join(f"{n}={s['curves'][n][ti]:.4g}" for n in VALUES), fontsize=10)
        pr = s["profiles"][th]
        bot.bar(np.arange(len(pr)), pr, width=1.0)
        plt.tight_layout(); plt.show()

    display(W.interactive(
        _show,
        ci=W.IntSlider(min=0, max=len(sweep) - 1, value=0, description="cluster"),
        ti=W.IntSlider(min=0, max=len(thetas) - 1, value=len(thetas) // 2, description="theta idx")))
except ImportError:
    print("ipywidgets not installed - use the grid above")

## 8 - Value vs theta, per cluster

In [ ]:
for ci, s in enumerate(sweep):
    if not s["vectors"]:
        continue
    e = s["entry"]
    fig, axes = plt.subplots(len(VALUES), 1, figsize=(11, 2.4 * len(VALUES)), squeeze=False)
    fig.suptitle(f"#{ci}  {e.text!r}   expected_rotation={e.expected_rotation}", fontsize=10)
    for ax, name in zip(axes[:, 0], VALUES):
        arr = s["curves"][name]
        fin = np.isfinite(arr)
        ax.plot(thetas[fin], arr[fin], marker=".", lw=1)
        if fin.any():
            opt = OPT.get(name, "max")
            ti = (np.nanargmax(np.where(fin, arr, -np.inf)) if opt == "max"
                  else np.nanargmin(np.where(fin, arr, np.inf)))
            ax.axvline(thetas[ti], color="tab:green", ls="--",
                       label=f"arg{opt} theta={thetas[ti]:+.1f}")
        if e.expected_rotation is not None:
            ax.axvline(e.expected_rotation, color="tab:red", ls=":", label="expected_rotation")
        ax.set_title(f"{name}   ({OPT.get(name, 'max')})", fontsize=9)
        ax.set_xlabel("theta (deg)"); ax.legend(fontsize=8)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## 9 - Rotated-bbox dimensions vs theta

`s["dims"]` is `(n_theta, 2)` = rotated-bbox **width, height** at each theta. Width min /
height min / area min all tend to sit at the deskew angle for a single text line.

In [ ]:
for ci, s in enumerate(sweep):
    if not s["vectors"]:
        continue
    e = s["entry"]
    w, h = s["dims"][:, 0], s["dims"][:, 1]
    area = w * h
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(thetas, w, marker=".", lw=1, label="width")
    ax.plot(thetas, h, marker=".", lw=1, label="height")
    ax.plot(thetas, area / max(np.nanmax(area), 1e-9) * np.nanmax(h),
            lw=1, ls="--", color="grey", label="area (scaled)")
    for arr, name, col in ((w, "min width", "tab:blue"), (h, "min height", "tab:orange"),
                           (area, "min area", "grey")):
        ti = int(np.nanargmin(arr))
        ax.axvline(thetas[ti], color=col, ls=":", lw=1,
                   label=f"{name} theta={thetas[ti]:+.1f}")
    if e.expected_rotation is not None:
        ax.axvline(e.expected_rotation, color="tab:red", lw=1.5, label="expected_rotation")
    ax.set_title(f"#{ci}  {e.text!r}   expected_rotation={e.expected_rotation}", fontsize=10)
    ax.set_xlabel("theta (deg)"); ax.set_ylabel("pt"); ax.legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.show()

    try:
        import pandas as pd
        d = pd.DataFrame({"theta": thetas, "width": w, "height": h, "area": area,
                          "n_lines": [len(s["profiles"][float(t)]) for t in thetas]})
        print(f"#{ci} {e.text!r}"); print(d.to_string(index=False)); print()
    except ImportError:
        pass